<a href="https://colab.research.google.com/github/ArmandoBarrios/unidad-3-/blob/main/practica_6_unidad_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
#Carga el Dataset
from google.colab import drive
drive.mount('/content/drive')
#ID del archivo
#https:https:https://drive.google.com/file/d/1V5ANwclpm0o4aHH_HvFeD0P0db0rqZOO/view?usp=sharing
file_id = '1V5ANwclpm0o4aHH_HvFeD0P0db0rqZOO'
url =  f"https://drive.google.com/uc?id={file_id}"
# Importamos librerias necesarias para Random Forest Classifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
prestamos_df = pd.read_csv(url)
# Eliminamos filas con valores nulos para evitar errores en el modelo
prestamos_df = prestamos_df.dropna(subset=['funded_amnt', 'int_rate', 'grade', 'purpose', 'addr_state', 'home_ownership', 'annual_inc', 'dti', 'revol_util', 'pub_rec_bankruptcies', 'loan_status'])
prestamos_df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag,debt_settlement_flag_date
0,2400,2400,2400.0,36 months,15.96,84.33,C,C5,NaN,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
1,10000,10000,10000.0,36 months,13.49,339.31,C,C1,AIR RESOURCES BOARD,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
2,3000,3000,3000.0,36 months,18.64,109.43,E,E1,MKC Accounting,9 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
3,5600,5600,5600.0,60 months,21.28,152.39,F,F2,NaN,4 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
4,5375,5375,5350.0,60 months,12.69,121.45,B,B5,Starbucks,< 1 year,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN


In [6]:
X = prestamos_df[['funded_amnt', 'int_rate', 'grade', 'purpose', 'addr_state',
'home_ownership', 'annual_inc', 'dti', 'revol_util',
'pub_rec_bankruptcies']]

# Convertimos variables categóricas a numéricas usando One-Hot Encoding antes del split
X = pd.get_dummies(X, drop_first=True)

# Variable objetivo o variable a predecir
y = prestamos_df['loan_status']

In [7]:
# Dividimos el dataFrame usando el X ya codificado
# stratify es para que mantenga la misma proporción de clases en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size= 0.4, stratify=y)

In [8]:
# verificamos la cantidad de registros asignados al dataframe de entrenamiento
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((11745, 33), (7831, 33), (11745,), (7831,))

In [9]:
# --- Normalización de variables --
# Ahora X_train contiene solo números gracias al encoding previo
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [10]:

# Verificar si hay valores nulos en el conjunto escalado
nan_count = np.isnan(X_train_scaled).sum()
print(f'Total de valores NaN en X_train_scaled: {nan_count}')

if nan_count > 0:
    print("Aún existen valores nulos en los datos.")
else:
    print("No se encontraron valores nulos.")

Total de valores NaN en X_train_scaled: 0
No se encontraron valores nulos.


In [11]:

# --- Limpieza final de NaNs antes del entrenamiento ---
import numpy as np

# Identificamos las filas que no tienen ningún NaN en X_train_scaled
mask_train = ~np.isnan(X_train_scaled).any(axis=1)
X_train_clean = X_train_scaled[mask_train]
y_train_clean = y_train[mask_train]

# --- Definimos el modelo SVM --
svm_model = SVC(
kernel='rbf',
# radial basis function (no lineal)
C=1.0,
# penalización por errores
gamma='scale',
# ajuste automático de gamma
class_weight='balanced',  # útil si hay desbalance
probability=True,
# permite obtener probabilidades
random_state=42
)

# --- Entrenamiento --
print("Entrenando modelo...")
svm_model.fit(X_train_clean, y_train_clean)
print("Entrenamiento completado.")

Entrenando modelo...
Entrenamiento completado.


In [12]:
# --- Limpieza de NaNs en Test ---
import numpy as np

# Identificamos filas sin NaNs en el conjunto de prueba
mask_test = ~np.isnan(X_test_scaled).any(axis=1)
X_test_clean = X_test_scaled[mask_test]
y_test_clean = y_test[mask_test]

# --- Predicción --
y_pred_svm = svm_model.predict(X_test_clean)

print("Reporte de Clasificación (SVM):")
print(classification_report(y_test_clean, y_pred_svm, target_names=["No Pagado", "Pagado"]))
print("\nMatriz de Confusión (SVM):")
print(confusion_matrix(y_test_clean, y_pred_svm))

Reporte de Clasificación (SVM):
              precision    recall  f1-score   support

   No Pagado       0.22      0.58      0.31      1148
      Pagado       0.90      0.64      0.75      6683

    accuracy                           0.63      7831
   macro avg       0.56      0.61      0.53      7831
weighted avg       0.80      0.63      0.68      7831


Matriz de Confusión (SVM):
[[ 665  483]
 [2416 4267]]


In [13]:
##  Probabilidades de la clase positiva ##
from sklearn.metrics import roc_curve, roc_auc_score
import plotly.graph_objects as go
# --- Probabilidades de la clase positiva ("Pagado" = 1) ---
# Usamos X_test_clean para evitar el error de valores NaN
y_prob_svm = svm_model.predict_proba(X_test_clean)[:, 1]
# --- Calcular AUC-ROC --
# Usamos y_test_clean para que coincida con las dimensiones de y_prob_svm
auc = roc_auc_score(y_test_clean, y_prob_svm)
print(f"\n AUC-ROC: {auc:.3f}")
# --- Curva ROC --
fpr, tpr, thresholds = roc_curve(y_test_clean, y_prob_svm, pos_label='Fully Paid')
# --- Grafica interactiva con Plotly --
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name='SVM', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Aleatorio', line=dict(dash='dash')))
fig.update_layout(
title=f'Probabilidades de la clase positiva - Curva ROC (AUC = {auc:.3f})',
xaxis_title='Tasa de Falsos Positivos (1 - Especificidad)',
yaxis_title='Tasa de Verdaderos Positivos (Sensibilidad)',
width=700, height=500
)
fig.show()



 AUC-ROC: 0.656


In [14]:
# --- Probabilidades de la clase negativa ("Pagado" = 0) --
# Usamos X_test_clean porque X_test_scaled contiene valores NaN
y_prob_svm = svm_model.predict_proba(X_test_clean)[:, 0]

# --- Calcular AUC-ROC --
# Usamos y_test_clean para que coincida con las dimensiones de la predicción
auc = roc_auc_score(y_test_clean, y_prob_svm)
print(f"\nAUC-ROC: {auc:.3f}")

# --- Curva ROC --
# Especificamos pos_label=0 (o la etiqueta correspondiente a impago) para la clase negativa
fpr, tpr, thresholds = roc_curve(y_test_clean, y_prob_svm, pos_label='Charged Off')

# --- Grafica interactiva con Plotly --
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name='SVM (Clase Negativa)', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Aleatorio', line=dict(dash='dash')))
fig.update_layout(
title=f'Probabilidades de la clase negativa - Curva ROC (AUC = {auc:.3f})',
xaxis_title='Tasa de Falsos Positivos (1 - Especificidad)',
yaxis_title='Tasa de Verdaderos Positivos (Sensibilidad)',
width=700, height=500
)
fig.show()


AUC-ROC: 0.344


In [15]:
# ===============================================
# Curvas ROC para ambas clases (positiva y negativa)
# ===============================================
from sklearn.metrics import roc_curve, roc_auc_score
import plotly.graph_objects as go

# --- Probabilidades predichas por el modelo --
# Usamos X_test_clean para evitar errores de NaN
y_prob = svm_model.predict_proba(X_test_clean)

# Clase positiva (Pagado)
fpr_pos, tpr_pos, _ = roc_curve(y_test_clean, y_prob[:, 1], pos_label='Fully Paid')
auc_pos = roc_auc_score(y_test_clean == 'Fully Paid', y_prob[:, 1])

# Clase negativa (No Pagado)
fpr_neg, tpr_neg, _ = roc_curve(y_test_clean, y_prob[:, 0], pos_label='Charged Off')
auc_neg = roc_auc_score(y_test_clean == 'Charged Off', y_prob[:, 0])

# --- Graficar con Plotly --
fig = go.Figure()
fig.add_trace(go.Scatter(
x=fpr_pos, y=tpr_pos,
mode='lines',
name=f'Clase positiva (Pagado) - AUC = {auc_pos:.3f}',
line=dict(color='blue', width=2)
))
fig.add_trace(go.Scatter(
x=fpr_neg, y=tpr_neg,
mode='lines',
name=f'Clase negativa (No Pagado) - AUC = {auc_neg:.3f}',
line=dict(color='green', width=2)
))
# Línea diagonal (modelo aleatorio)
fig.add_trace(go.Scatter(
x=[0, 1], y=[0, 1],
mode='lines',
name='Aleatorio',
line=dict(color='red', width=1, dash='dash')
))
# --- Personalización del gráfico --
fig.update_layout(
title="Comparación de Curvas ROC - Clases Positiva y Negativa",
xaxis_title="Tasa de Falsos Positivos (1 - Especificidad)",
yaxis_title="Tasa de Verdaderos Positivos (Sensibilidad)",
width=900, height=600,
legend=dict(x=0.6, y=0.05)
)
fig.show()


In [16]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, fbeta_score, classification_report, confusion_matrix
import numpy as np
# Definimos el F2-score como métrica personalizada (priorizando recall sobre precisión)
# Usamos 'Charged Off' como pos_label porque es la clase de interés (impagos)
f2_scorer = make_scorer(fbeta_score, beta=2, pos_label='Charged Off')
# Modelo base
svm = SVC(probability=False, class_weight='balanced', random_state=42)
# Definimos los hiperparámetros a explorar
param_grid = {
'C': [0.1, 1, 10],
'kernel': ['linear', 'rbf', 'poly'],
'gamma': ['scale', 'auto'],
'degree': [2, 3],  # solo afecta al kernel 'poly'
}
# Configuración del GridSearchCV
grid_search = GridSearchCV(
estimator=svm,
param_grid=param_grid,
scoring=f2_scorer,  # optimizamos el F2-score
cv=5,
verbose=2,
n_jobs=-1
)
# Entrenamiento usando datos LIMPIOS
print("Entrenando GridSearchCV para SVM con datos limpios...\n")
grid_search.fit(X_train_clean, y_train_clean)
# Resultados
print("\n Búsqueda finalizada.")
print("Mejores hiperparámetros encontrados:")
print(grid_search.best_params_)
print("\nMejor puntaje promedio (F2-score):")
print(round(grid_search.best_score_, 4))
# Evaluación con los mejores parámetros usando datos LIMPIOS de test
svm_optimo = grid_search.best_estimator_
y_pred_opt = svm_optimo.predict(X_test_clean)
print("\n Reporte de Clasificación (SVM optimizado para impagos):\n")
print(classification_report(y_test_clean, y_pred_opt, target_names=["No Pagado", "Pagado"]))
print("Matriz de Confusión:")
print(confusion_matrix(y_test_clean, y_pred_opt))

Entrenando GridSearchCV para SVM con datos limpios...

Fitting 5 folds for each of 36 candidates, totalling 180 fits


KeyboardInterrupt: 